# 01 · Data profiling
**Business question:** before building any planning metric, what does the data actually contain, and where are its traps?

Covers the calendar (events, SNAP days), prices (launch dates, assortment growth) and sales (volume mix, zero-sales days, missing prices).

> **Run one notebook at a time.** DuckDB allows only one open connection to `practice.duckdb`. Before opening another notebook, use *Kernel → Shut Down Kernel* in this one.

In [ ]:
import duckdb

# Connect to the local DuckDB file built in 00_setup_and_load
con = duckdb.connect('../data/practice.duckdb')

# Memory settings: needed for the 59M-row sales_daily table on a laptop
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

## 1. Calendar

In [ ]:
# Date range covered by the dataset
con.sql("""
SELECT MIN(date) AS first_day,
       MAX(date) AS last_day
FROM calendar
""").df()

,min(date),max(date)
0,2011-01-29,2016-06-19


In [ ]:
# SNAP days per year in California
# SNAP = US food-assistance payment days, a known demand driver for food
con.sql("""
SELECT year,
       SUM(snap_CA) AS snap_days
FROM calendar
GROUP BY year
ORDER BY year
""").df()

,year,snap_days
0,2011,110.0
1,2012,120.0
2,2013,120.0
3,2014,120.0
4,2015,120.0
5,2016,60.0


In [ ]:
# Event days by event type, flagged Large if more than 40 days
con.sql("""
SELECT event_type_1,
       COUNT(event_type_1) AS event_days,
       MIN(date)           AS first_seen,
       CASE WHEN COUNT(event_type_1) > 40 THEN 'Large' ELSE 'Small' END AS size
FROM calendar
WHERE event_type_1 IS NOT NULL
GROUP BY event_type_1
ORDER BY event_days DESC
""").df()

,event_type_1,event_days,first_seen,size
0,Religious,55,2011-03-09,Large
1,National,52,2011-02-21,Large
2,Cultural,37,2011-02-14,Small
3,Sporting,18,2011-02-06,Small


In [ ]:
# Which weekdays carry events
# COUNT(event_name_1) skips NULLs, so it counts only days with an event
con.sql("""
SELECT weekday,
       COUNT(weekday)      AS total_days,
       COUNT(event_name_1) AS event_days
FROM calendar
GROUP BY weekday
ORDER BY event_days DESC
""").df()

,weekday,total_days,event_days
0,Sunday,282,40
1,Monday,281,37
2,Thursday,281,28
3,Wednesday,281,21
4,Tuesday,281,17
5,Saturday,282,11
6,Friday,281,8


**Findings: calendar**
- 1,969 days, from 29 Jan 2011 to 19 Jun 2016 (2011 and 2016 are partial years).
- California has 10 SNAP days a month (120 a year), a recurring demand driver for food items.
- 162 event days: Religious 55, National 52, Cultural 37, Sporting 18.
- Events cluster on Sundays and Mondays (holidays often move to Mondays), so weekday and event effects overlap. A forecast model needs both features.

## 2. Prices

In [ ]:
# Overall profile of the price table
con.sql("""
SELECT COUNT(*)                 AS number_of_rows,
       COUNT(DISTINCT store_id) AS distinct_stores,
       COUNT(DISTINCT item_id)  AS distinct_items,
       MIN(sell_price)          AS min_sell_price,
       MAX(sell_price)          AS max_sell_price,
       AVG(sell_price)          AS avg_sell_price
FROM prices
""").df()

,number_of_rows,distict_stores,item_id,min_sell_price,max_sell_price,avg_sell_price
0,6841121,10,3049,0.01,107.32,4.410952


In [ ]:
# Days with no price for one item-store, HOBBIES_1_001 in CA_1
# The item/store filter sits in ON (not WHERE) so the LEFT JOIN keeps every calendar day
con.sql("""
SELECT COUNT(c.date) AS days_without_price
FROM calendar c
LEFT JOIN prices p
       ON  c.wm_yr_wk = p.wm_yr_wk
       AND p.item_id  = 'HOBBIES_1_001'
       AND p.store_id = 'CA_1'
WHERE p.sell_price IS NULL
""").df()

,count(c.date)
0,896


In [ ]:
# Assortment growth: share of the catalog with a price, by year and store
# 3049 = total distinct items (from the profile above)
con.sql("""
SELECT c.year                    AS year,
       p.store_id                AS stores,
       COUNT(DISTINCT p.item_id) AS distinct_prod,
       ROUND(COUNT(DISTINCT p.item_id) * 100.0 / 3049, 1) AS pct_sales_prod
FROM calendar c
INNER JOIN prices p
        ON c.wm_yr_wk = p.wm_yr_wk
WHERE p.sell_price IS NOT NULL
GROUP BY c.year, p.store_id
ORDER BY pct_sales_prod
""").df()

,year,stores,distinct_prod,pct_sales_prod
0,2011,WI_1,1430,46.9
1,2011,WI_2,1497,49.1
2,2011,CA_2,1548,50.8
3,2011,CA_4,1675,54.9
4,2011,TX_3,1732,56.8
5,2011,CA_3,1750,57.4
6,2011,WI_3,1765,57.9
7,2011,CA_1,1781,58.4
8,2011,TX_1,1791,58.7
9,2011,TX_2,1793,58.8


In [ ]:
# Monthly price stability for the top-selling item FOODS_3_090 in CA_1, 2015
con.sql("""
SELECT MONTH(c.date)     AS month,
       MIN(p.sell_price) AS lowest_sell_price,
       MAX(p.sell_price) AS highest_sell_price,
       CASE WHEN lowest_sell_price = highest_sell_price THEN 'Price Stable'
            ELSE 'Price Changed' END AS price_move
FROM calendar c
INNER JOIN prices p
        ON c.wm_yr_wk = p.wm_yr_wk
WHERE p.store_id = 'CA_1'
  AND p.item_id  = 'FOODS_3_090'
  AND YEAR(c.date) = 2015
GROUP BY MONTH(c.date)
ORDER BY month
""").df()

,month,lowest_sell_price,highest_sell_price,price_move
0,1,1.48,1.48,Price Stable
1,2,1.48,1.48,Price Stable
2,3,1.48,1.48,Price Stable
3,4,1.48,1.48,Price Stable
4,5,1.48,1.48,Price Stable
5,6,1.48,1.48,Price Stable
6,7,1.48,1.48,Price Stable
7,8,1.48,1.48,Price Stable
8,9,1.48,1.48,Price Stable
9,10,1.48,1.50,Price Changed


**Findings: prices**
- 6.84M price rows covering 10 stores and all 3,049 items; prices run from \$0.01 to \$107.32 (average \$4.41).
- A missing price means the item was **not on sale yet**: HOBBIES_1_001 had no price for its first 896 days (until July 2013).
- The assortment grew from roughly 47–59% of the catalog in 2011 to about 100% by 2015–16. Early history is thin for many items.
- FOODS_3_090 kept a stable price through 2015 (one small change in October), so price changes do not explain its sales swings.

## 3. Sales

In [ ]:
# Units and revenue by category
con.sql("""
SELECT cat_id                 AS category,
       SUM(units)             AS total_unit,
       ROUND(SUM(revenue), 0) AS total_revenue
FROM sales_daily
GROUP BY cat_id
ORDER BY total_revenue DESC
""").df()

,category,total_unit,total_revenue
0,FOODS,45922427.0,111140025.0
1,HOUSEHOLD,14764090.0,57115876.0
2,HOBBIES,6240656.0,23321644.0


In [ ]:
# Units and revenue by store
con.sql("""
SELECT state_id,
       store_id,
       SUM(units)             AS total_units,
       ROUND(SUM(revenue), 0) AS total_revenue
FROM sales_daily
GROUP BY state_id, store_id
ORDER BY total_revenue DESC
""").df()

,state_id,store_id,total_units,total_revenue
0,CA,CA_3,11363540.0,32699138.0
1,CA,CA_1,7832248.0,22954284.0
2,TX,TX_2,7329642.0,20893002.0
3,TX,TX_3,6205940.0,18190299.0
4,WI,WI_2,6697988.0,18131985.0
5,CA,CA_2,5818395.0,17848209.0
6,WI,WI_3,6542557.0,17250126.0
7,TX,TX_1,5692823.0,16037341.0
8,WI,WI_1,5261506.0,15107434.0
9,CA,CA_4,4182534.0,12465728.0


In [ ]:
# Share of item-store-days with zero sales, while the item was on sale
con.sql("""
SELECT cat_id,
       ROUND(SUM(CASE WHEN units = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_zero_days
FROM sales_daily
WHERE sell_price IS NOT NULL                -- exclude days before the item launched
GROUP BY cat_id
ORDER BY pct_zero_days DESC
""").df()

,cat_id,pct_zero_days
0,HOBBIES,71.5
1,HOUSEHOLD,64.6
2,FOODS,51.1


In [ ]:
# Share of all rows without a price (item not yet on sale)
# Note: reusing aliases in the same SELECT works in DuckDB only
con.sql("""
SELECT COUNT(*) AS total_days,
       SUM(CASE WHEN sell_price IS NULL THEN 1 ELSE 0 END) AS days_no_price,
       ROUND(days_no_price / total_days * 100, 1)          AS pct_no_price
FROM sales_daily
""").df()

,total_days,days_no_price,pct_no_price
0,59181090,12299413.0,20.8


**Findings: sales**
- FOODS is about 69% of units and 58% of revenue; CA_3 is the largest store (\$32.7M revenue, 17% of the total).
- Zero-sales days are very common even when the item is on sale: HOBBIES 71.5%, HOUSEHOLD 64.6%, FOODS 51.1%. Much of the catalog is **intermittent demand**, which inflates percentage-error metrics and argues for item segmentation (notebook 03).
- 20.8% of all rows have no price because the item was not on sale yet. These rows must be excluded from accuracy and feature calculations, otherwise they look like zero demand.